# Gymnasium — based reinforcement learning

**Goal:** You will **run controlled experiments** in reinforcement learning and reflect on the significance of the results.

✅ The RL code is provided.  
✅ Your work is to observe behavior (plots/roll-outs), and reflect on their significance.

---

## Learning objectives

- Understand why **exploration** is needed (ε-greedy).
- Contrast **planning** (Value Iteration, model-based) with **learning** (Q-learning, model-free).
- See how stochasticity and hyperparameters affect performance.
- Build intuition through **trajectories, policy arrows, Q-heatmaps, and roll-outs**.

---

## Important note

This week is conceptually different from supervised learning.  
**Please study the video lectures carefully before class** so the in-class discussion is productive.


## Fixed specs used for grading (do not change)

### Global seed
- `SEED = 123`

### Part A (Bandit)
- Bernoulli means:  
  `p = [0.05, 0.12, 0.18, 0.22, 0.27, 0.31, 0.33, 0.36, 0.38, 0.41]`
- Horizon: `T = 7000`
- We will run ε-greedy with ε in `{0.0, 0.2, 0.4}`
- Regret uses **true means**: \(\sum_t (p^* - p[a_t])\), where \(p^* = 0.41\)

### Part B (Gymnasium Gridworld)
- Coordinates: state `(x,y)` with `(0,0)` at **bottom-left**
- Observation: `sid = y*4 + x` (Discrete(16))
- `S = (0,3)`, `G = (3,0)`, lava: `(1,2)` and `(3,1)`
- Actions: `0=up, 1=right, 2=down, 3=left`
- Rewards:
  - reach goal: +10 (terminal)
  - reach lava: −10 (terminal)
  - otherwise: −1
- Discount: `gamma = 0.95`
- Wrappers:
  - `TimeLimit(max_episode_steps=50)`
  - `RecordEpisodeStatistics`

### Q-learning (training/eval)
- Train: 4000 episodes, α=0.1, ε=0.2, Q init 0
- Evaluate: 500 episodes, greedy (ε=0)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import gymnasium as gym
from gymnasium import spaces
from gymnasium.wrappers import TimeLimit, RecordEpisodeStatistics

# -------------------------
# Global constants
# -------------------------
SEED = 123

# Part A
BANDIT_P = np.array([0.05, 0.12, 0.18, 0.22, 0.27, 0.31, 0.33, 0.36, 0.38, 0.41], dtype=float)
BANDIT_T = 7000
BANDIT_EPS_LIST = [0.0, 0.2, 0.4]   # experimentation knob in Part A (keep fixed for grading)

# Part B (Gridworld)
GRID_N = 4
START = (0, 3)
GOAL = (3, 0)
LAVAS = {(1, 2), (3, 1)}
GAMMA = 0.95

# Q-learning
QL_EPISODES = 4000
QL_ALPHA = 0.1
QL_EPS = 0.2
EVAL_EPISODES = 500


## Part A — Bandits

In [ ]:
# ============================================================
# Part A — Bandits (ε-greedy)
# ============================================================
# Bandit = no state, only action choice + stochastic reward.
# We sweep ε to see the exploration trade-off numerically.
# ============================================================

class EpsGreedyBanditAgent:
    """ε-greedy for Bernoulli bandits.

    Notes:
    - Random tie-breaking for argmax avoids bias.
    - Regret uses TRUE means (p), not sampled rewards.
    """

    def __init__(self, epsilon: float, seed: int):
        self.epsilon = float(epsilon)
        self.rng = np.random.default_rng(seed)

    def _argmax_random_tie(self, values: np.ndarray) -> int:
        maxv = np.max(values)
        idx = np.flatnonzero(values == maxv)
        return int(idx[0]) if len(idx) == 1 else int(self.rng.choice(idx))

    def fit(self, p: np.ndarray, T: int):
        K = len(p)
        q = np.zeros(K, dtype=float)   # empirical means
        n = np.zeros(K, dtype=int)     # pull counts

        p_star = float(np.max(p))
        a_star = int(np.argmax(p))

        cum_reward = 0.0
        cum_regret = 0.0
        regret_curve = np.zeros(T, dtype=float)
        optimal_rate_curve = np.zeros(T, dtype=float)
        optimal_picks = 0

        for t in range(1, T + 1):
            # Choose action
            if self.rng.random() < self.epsilon:
                a = int(self.rng.integers(0, K))        # explore
            else:
                a = self._argmax_random_tie(q)          # exploit

            # Sample Bernoulli reward
            r = 1.0 if (self.rng.random() < p[a]) else 0.0

            # Update estimates
            n[a] += 1
            q[a] += (r - q[a]) / n[a]

            # Bookkeeping
            cum_reward += r
            cum_regret += (p_star - p[a])

            if a == a_star:
                optimal_picks += 1

            regret_curve[t-1] = cum_regret
            optimal_rate_curve[t-1] = optimal_picks / t

        return float(cum_reward), float(cum_regret), regret_curve, optimal_rate_curve


def run_bandit_sweep(p: np.ndarray, T: int, eps_list: list[float], seed: int):
    """Run ε-greedy for multiple ε values and return a dict of results."""
    results = {}
    for eps in eps_list:
        agent = EpsGreedyBanditAgent(epsilon=eps, seed=seed)
        cum_reward, cum_regret, reg_curve, opt_curve = agent.fit(p, T)
        results[eps] = {
            "cum_reward": cum_reward,
            "cum_regret": cum_regret,
            "reg_curve": reg_curve,
            "opt_curve": opt_curve,
        }
    return results


### Part A experiments

In [ ]:
# =========================
# Part A — Experiments
# =========================

bandit_results = run_bandit_sweep(BANDIT_P, BANDIT_T, BANDIT_EPS_LIST, SEED)

# Plot cumulative regret for different epsilons
t = np.arange(1, BANDIT_T + 1)
plt.figure()
for eps in BANDIT_EPS_LIST:
    plt.plot(t, bandit_results[eps]["reg_curve"], label=f"ε={eps}")
plt.xlabel("t")
plt.ylabel("Cumulative regret")
plt.title("Bandit: ε-greedy regret vs ε")
plt.legend()
plt.show()

# Plot optimal-arm selection rate
plt.figure()
for eps in BANDIT_EPS_LIST:
    plt.plot(t, bandit_results[eps]["opt_curve"], label=f"ε={eps}")
plt.xlabel("t")
plt.ylabel("Running fraction of optimal-arm picks")
plt.title("Bandit: optimal-arm selection rate vs ε")
#plt.ylim(0, 1.0)
plt.legend()
plt.show()

# Quick textual summary
for eps in BANDIT_EPS_LIST:
    print(f"ε={eps}: cum_regret={bandit_results[eps]['cum_regret']:.3f}, final optimal-rate={bandit_results[eps]['opt_curve'][-1]:.3f}")


## Part B — MDP Gridworld (Gymnasium)

Environment + wrappers. Read the markup comments carefully.

In [ ]:
# ============================================================
# Part B — Gymnasium Gridworld (MDP)
# ============================================================
# Gymnasium API:
#   reset(seed=...) -> (obs, info)
#   step(a) -> (obs, reward, terminated, truncated, info)
#
# We use wrappers for "standardized" behavior:
#   TimeLimit(max_episode_steps=50) handles truncation
#   RecordEpisodeStatistics stores episode return/length in info
# ============================================================

class Gridworld4x4(gym.Env):
    metadata = {"render_modes": ["ansi"]}

    def __init__(self, stochastic: bool):
        super().__init__()
        self.stochastic = bool(stochastic)

        self.n = GRID_N
        self.action_space = spaces.Discrete(GRID_N)
        self.observation_space = spaces.Discrete(self.n * self.n)

        self.state = None  # (x,y)

    # ---------- helpers ----------
    def _state_id(self, xy: tuple[int,int]) -> int:
        x, y = xy
        return y * self.n + x

    def _id_to_xy(self, sid: int) -> tuple[int,int]:
        y = sid // self.n
        x = sid % self.n
        return (x, y)

    def _is_terminal(self, xy: tuple[int,int]) -> bool:
        return (xy == GOAL) or (xy in LAVAS)

    def _reward(self, next_xy: tuple[int,int]) -> float:
        if next_xy == GOAL:
            return 10.0
        if next_xy in LAVAS:
            return -10.0
        return -1.0

    def _move_det(self, xy: tuple[int,int], a: int) -> tuple[int,int]:
        # 0=up,1=right,2=down,3=left
        x, y = xy
        if a == 0:
            x2, y2 = x, y + 1
        elif a == 1:
            x2, y2 = x + 1, y
        elif a == 2:
            x2, y2 = x, y - 1
        elif a == 3:
            x2, y2 = x - 1, y
        else:
            raise ValueError("bad action")

        if 0 <= x2 < self.n and 0 <= y2 < self.n:
            return (x2, y2)
        return (x, y)

    def _perps(self, a: int) -> tuple[int,int]:
        if a in (0, 2):
            return (3, 1)
        if a in (1, 3):
            return (0, 2)
        raise ValueError("bad action")

    # ---------- Gymnasium API ----------
    def reset(self, *, seed: int | None = None, options=None):
        super().reset(seed=seed)  # sets self.np_random
        self.state = START
        return self._state_id(self.state), {}

    def step(self, action: int):
        if self._is_terminal(self.state):
            return self._state_id(self.state), 0.0, True, False, {}

        a_exec = int(action)

        if self.stochastic:
            u = float(self.np_random.random())
            if u < 0.8:
                a_exec = a_exec
            else:
                p1, p2 = self._perps(a_exec)
                a_exec = p1 if u < 0.9 else p2

        next_xy = self._move_det(self.state, a_exec)
        r = self._reward(next_xy)
        terminated = self._is_terminal(next_xy)
        truncated = False  # TimeLimit handles truncation
        self.state = next_xy

        return self._state_id(self.state), r, terminated, truncated, {}

    def render(self):
        rows = []
        for y in reversed(range(self.n)):
            row = []
            for x in range(self.n):
                xy = (x, y)
                if xy == self.state:
                    row.append("A")
                elif xy == START:
                    row.append("S")
                elif xy == GOAL:
                    row.append("G")
                elif xy in LAVAS:
                    row.append("L")
                else:
                    row.append(".")
            rows.append(" ".join(row))
        return "\n".join(rows)


def make_env(stochastic: bool, seed: int):
    env = Gridworld4x4(stochastic=stochastic)
    env = TimeLimit(env, max_episode_steps=50)
    env = RecordEpisodeStatistics(env)
    env.reset(seed=seed)
    return env


## Planning vs learning

Value Iteration (planning) vs Q-learning (learning). Both are given as small classes.

In [ ]:
# ============================================================
# Model-based planning: Value Iteration
# ============================================================

class ValueIterationPlanner:
    def __init__(self, gamma: float = GAMMA, tol: float = 1e-8):
        self.gamma = float(gamma)
        self.tol = float(tol)

    def solve(self, env: Gridworld4x4):
        assert env.stochastic is False, "Value iteration assumes deterministic transitions here."
        nS = env.observation_space.n
        nA = env.action_space.n

        terminal_ids = {sid for sid in range(nS) if env._is_terminal(env._id_to_xy(sid))}
        V = np.zeros(nS, dtype=float)

        iters = 0
        while True:
            iters += 1
            V_new = V.copy()
            delta = 0.0

            for sid in range(nS):
                if sid in terminal_ids:
                    V_new[sid] = 0.0
                    continue

                xy = env._id_to_xy(sid)
                best = -np.inf
                for a in range(nA):
                    xy2 = env._move_det(xy, a)
                    r = env._reward(xy2)
                    sid2 = env._state_id(xy2)
                    val = r + self.gamma * (0.0 if sid2 in terminal_ids else V[sid2])
                    best = max(best, val)

                V_new[sid] = best
                delta = max(delta, abs(V_new[sid] - V[sid]))

            V = V_new
            if delta < self.tol:
                break
            if iters > 100000:
                raise RuntimeError("Value iteration did not converge.")

        return V, iters


In [ ]:
# ============================================================
# Model-free learning: Q-learning (class wrapper)
# ============================================================

class QLearningAgent:
    def __init__(self, alpha: float = QL_ALPHA, epsilon: float = QL_EPS, gamma: float = GAMMA, seed: int = SEED):
        self.alpha = float(alpha)
        self.epsilon = float(epsilon)
        self.gamma = float(gamma)
        self.rng = np.random.default_rng(seed)
        self.Q = None

    def _argmax_random_tie(self, values: np.ndarray) -> int:
        maxv = np.max(values)
        idx = np.flatnonzero(values == maxv)
        return int(idx[0]) if len(idx) == 1 else int(self.rng.choice(idx))

    def _eps_greedy(self, q_row: np.ndarray, epsilon: float) -> tuple[int, bool]:
        if self.rng.random() < epsilon:
            return int(self.rng.integers(0, len(q_row))), True
        return self._argmax_random_tie(q_row), False

    def fit(self, env, episodes: int = QL_EPISODES):
        nS = env.observation_space.n
        nA = env.action_space.n
        self.Q = np.zeros((nS, nA), dtype=float)

        ep_returns = np.zeros(episodes, dtype=float)
        ep_explore_frac = np.zeros(episodes, dtype=float)
        visit_counts = np.zeros(nS, dtype=int)

        for ep in range(episodes):
            obs, _ = env.reset()
            G = 0.0
            disc = 1.0
            explore_ct = 0
            act_ct = 0

            while True:
                visit_counts[obs] += 1
                a, explored = self._eps_greedy(self.Q[obs], self.epsilon)
                explore_ct += int(explored)
                act_ct += 1

                obs2, r, terminated, truncated, info = env.step(a)
                done = terminated or truncated

                target = r if done else (r + self.gamma * np.max(self.Q[obs2]))
                self.Q[obs, a] += self.alpha * (target - self.Q[obs, a])

                G += disc * r
                disc *= self.gamma
                obs = obs2

                if done:
                    break

            ep_returns[ep] = G
            ep_explore_frac[ep] = explore_ct / act_ct if act_ct else 0.0

        return ep_returns, ep_explore_frac, visit_counts

    def evaluate(self, env, episodes: int = EVAL_EPISODES, seed: int = SEED) -> tuple[float, float]:
        env.reset(seed=seed)
        successes = 0
        returns = []

        for _ in range(episodes):
            obs, _ = env.reset()
            G = 0.0
            disc = 1.0
            final_xy = None

            while True:
                a = self._argmax_random_tie(self.Q[obs])
                obs2, r, terminated, truncated, info = env.step(a)
                G += disc * r
                disc *= self.gamma
                obs = obs2

                if terminated or truncated:
                    base_env = env
                    while hasattr(base_env, "env"):
                        base_env = base_env.env
                    final_xy = base_env._id_to_xy(obs)
                    break

            if final_xy == GOAL:
                successes += 1
            returns.append(G)

        return successes / episodes, float(np.mean(returns))


## Visual intuition tools

Policy arrows, Q heatmaps, and ASCII roll-outs.

In [ ]:
# ============================================================
# Visualization helpers (for intuition)
# ============================================================
# Key improvement (v5.1):
# - We MASK arrows for states that were never visited during training,
#   because Q(s,a) is undefined there (still at initialization).
# - We annotate each arrow with the greedy state's value: max_a Q(s,a).
#   This is NOT V*(s) from value iteration; it is "value under learned Q".
# ============================================================

def plot_policy_arrows_with_values(
    ax,
    policy: np.ndarray,
    Q: np.ndarray,
    mask: np.ndarray | None = None,
    title: str = "",
    value_fmt: str = "{:.2f}",
):
    """Plot greedy policy arrows, optionally only on masked (visited) states,
    and annotate each state with v(s)=max_a Q(s,a)."""

    ax.set_title(title)
    ax.set_xlim(-0.5, GRID_N - 0.5)
    ax.set_ylim(-0.5, GRID_N - 0.5)
    ax.set_xticks(range(GRID_N))
    ax.set_yticks(range(GRID_N))
    ax.grid(True)
    ax.set_aspect("equal")

    # markers
    sx, sy = START
    gx, gy = GOAL
    ax.scatter([sx], [sy], marker="s", s=200)
    ax.scatter([gx], [gy], marker="*", s=250)
    for (lx, ly) in LAVAS:
        ax.scatter([lx], [ly], marker="X", s=200)

    # action: 0 up, 1 right, 2 down, 3 left
    arrows = {0:(0.0, 0.32), 1:(0.32, 0.0), 2:(0.0, -0.32), 3:(-0.32, 0.0)}
    head = 0.10

    for sid, a in enumerate(policy):
        if mask is not None and not bool(mask[sid]):
            continue

        x = sid % GRID_N
        y = sid // GRID_N
        xy = (x, y)

        # Skip terminals: arrows + values are less meaningful here
        if xy == GOAL or xy in LAVAS:
            continue

        dx, dy = arrows[int(a)]
        ax.arrow(x, y, dx, dy, head_width=head, length_includes_head=True)

        v = float(np.max(Q[sid]))
        ax.text(
            x,
            y - 0.25,
            value_fmt.format(v),
            ha="center",
            va="top",
            fontsize=9,
        )

def plot_q_action_heatmaps(Q: np.ndarray, title_prefix: str = "Q"):
    fig, axes = plt.subplots(1, 4, figsize=(12, 3))
    for a in range(4):
        ax = axes[a]
        im = ax.imshow(Q[:, a].reshape(GRID_N, GRID_N).T, origin="lower")
        ax.set_title(f"{title_prefix}(s,a={a})")
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.show()

def text_rollout(env, Q: np.ndarray, rng: np.random.Generator, max_print_steps: int = 25, seed: int = SEED):
    env.reset(seed=seed)
    obs, _ = env.reset()
    for t in range(max_print_steps):
        print(f"t={t}")
        base_env = env
        while hasattr(base_env, "env"):
            base_env = base_env.env
        print(base_env.render())

        maxv = np.max(Q[obs])
        idx = np.flatnonzero(Q[obs] == maxv)
        a = int(rng.choice(idx))
        obs, _, terminated, truncated, _ = env.step(a)
        if terminated or truncated:
            print(f"terminal={terminated}, truncated={truncated}")
            base_env = env
            while hasattr(base_env, "env"):
                base_env = base_env.env
            print(base_env.render())
            break


### Part B experiments and visuals

In [ ]:
# =========================
# Part B — Experiments & visuals (not graded directly)
# =========================

# Planning (Value Iteration)
base_det = Gridworld4x4(stochastic=False)
V, iters = ValueIterationPlanner(gamma=GAMMA, tol=1e-8).solve(base_det)
print("Value Iteration done. V*(S) =", V[START[1]*GRID_N + START[0]], "iters =", iters)




# Q-learning (deterministic)
env_det = make_env(stochastic=False, seed=SEED)
ql_det = QLearningAgent(seed=SEED)
det_returns, det_explore, det_visits = ql_det.fit(env_det, episodes=QL_EPISODES)

'''
plt.figure()
plt.plot(np.arange(1, QL_EPISODES+1), det_returns)
plt.xlabel("Training episode")
plt.ylabel("Discounted return")
plt.title("Q-learning training returns (deterministic)")
plt.show()
'''

# Mask arrows/values to states that were actually visited during training
policy_det = np.argmax(ql_det.Q, axis=1)
visited_det = det_visits > 0

plt.figure(figsize=(5,5))
plot_policy_arrows_with_values(
    plt.gca(),
    policy=policy_det,
    Q=ql_det.Q,
    mask=visited_det,
    title="Greedy policy arrows + v(s)=max_a Q(s,a) (visited states only) — deterministic",
)
plt.show()

'''
plot_q_action_heatmaps(ql_det.Q, title_prefix="Q_det")
'''

print("Greedy rollout (deterministic):")
text_rollout(make_env(False, SEED), ql_det.Q, rng=np.random.default_rng(SEED), seed=SEED)




# Q-learning (stochastic)
env_sto = make_env(stochastic=True, seed=SEED)
ql_sto = QLearningAgent(seed=SEED)
sto_returns, sto_explore, sto_visits = ql_sto.fit(env_sto, episodes=QL_EPISODES)

'''
plt.figure()
plt.plot(np.arange(1, QL_EPISODES+1), sto_returns)
plt.xlabel("Training episode")
plt.ylabel("Discounted return")
plt.title("Q-learning training returns (stochastic)")
plt.show()
'''

policy_sto = np.argmax(ql_sto.Q, axis=1)
visited_sto = sto_visits > 0

plt.figure(figsize=(5,5))
plot_policy_arrows_with_values(
    plt.gca(),
    policy=policy_sto,
    Q=ql_sto.Q,
    mask=visited_sto,
    title="Greedy policy arrows + v(s)=max_a Q(s,a) (visited states only) — stochastic",
)
plt.show()

'''
plot_q_action_heatmaps(ql_sto.Q, title_prefix="Q_sto")
'''

print("Greedy rollout (stochastic):")
text_rollout(make_env(True, SEED), ql_sto.Q, rng=np.random.default_rng(SEED), seed=SEED)


## Some outputs 

In [ ]:
# ==============
# Some outputs #
# ==============

# ----- Part A outputs -----
A1 = float(bandit_results[0.0]["cum_regret"])
A2 = float(bandit_results[0.2]["cum_regret"])
A3 = float(bandit_results[0.4]["cum_regret"])
A4 = float(int(np.argmin([A1, A2, A3])))  # best epsilon index
A5 = float(A1 - A2)                       # regret gap (0.0 vs 0.2)

# ----- Part B outputs -----
B1 = float(V[START[1]*GRID_N + START[0]])
B2 = float(iters)
B3, B4 = ql_det.evaluate(make_env(False, SEED), episodes=EVAL_EPISODES, seed=SEED)
B5, B6 = ql_sto.evaluate(make_env(True, SEED), episodes=EVAL_EPISODES, seed=SEED)

for x in [A1, A2, A3, A4, A5, B1, B2, B3, B4, B5, B6]:
    print(x)
